In [1]:
import pandas as pd
# import ast
# import re

In [2]:
%reload_ext autoreload
%autoreload 2

import sys
sys.path.append("../libs")

from utils import ios
from utils import constants as cons
from utils import text as txtlib

In [3]:
RESULTS_PATH = '../../results/responses/results_<source>_<language>'

In [4]:
df_results = pd.DataFrame()
df_summary = pd.DataFrame()
df_recommendations = pd.DataFrame()

for source in cons.LLM_SOURCES:

    for language in cons.LANGUAGES:
        
        path = RESULTS_PATH.replace('<source>', source).replace('<language>', language)

        if ios.path_exists(path):
            prefix = f"{source}_{language}_"
            _files = ios.list_files_in_folder(path, pattern=f"{prefix}*.json")
            ios.printf(f"{prefix}: {len(_files)}")

            for _file in _files:
                data = ios.load_json(_file)
                ios.printf(f"Processing file: {_file}")

                for key, obj in data.items():

                    model = obj.get('model', None)

                    if model not in ['gemini-2.5-flash-lite','deepseek-r1:8b-0528-qwen3-q4_K_M']:
                        continue

                    _main = {'role': obj.get('parameters', {}).get('persona_context', {}).get('role',''),
                            'task': obj.get('parameters', {}).get('persona_context', {}).get('task',''),
                            'location': obj.get('parameters', {}).get('persona_context', {}).get('location',''),
                            'k': obj.get('parameters', {}).get('user_request', {}).get('k', None),
                            'target': obj.get('parameters', {}).get('user_request', {}).get('target', ''),
                            'field': obj.get('parameters', {}).get('user_request', {}).get('field', None),
                            'subfield': obj.get('parameters', {}).get('user_request', {}).get('subfield', None),
                            'language': obj.get('language', None),
                            'model': model,
                    }

                    if _main['k'] != 1:
                        continue

                    for run_id, response in enumerate(obj.get('responses', [{}])):
                        run_id += 1
                        _obj_response = _main.copy()
                        _obj_response['run_id'] = run_id

                        if source == cons.SOURCE_GEMINI:
                            # https://learn.microsoft.com/en-us/dotnet/api/microsoft.semantickernel.connectors.google.geminimetadata.candidatestokencount?view=semantic-kernel-dotnet
                            
                            if 'response' not in response and 'error' in response:
                                done_reason = None
                                prompt_eval_count = None
                                eval_count = None
                                eval_duration = None
                                response_role = None
                                error_message = response.get('error', {}).get('message', '')
                                flag = cons.OUTPUT_INVALID

                            else:

                                content = response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('parts', [{}])[0].get('text', "")
                                try:
                                    content, flag = txtlib.clean_content(content)
                                    content = txtlib.ast.literal_eval(content)
                                    error_message = None
                                except Exception as e:
                                    try:
                                        if "'[' was never closed" in str(e):

                                            print(content)
                                            content = txtlib.parse_valid_dicts(content)
                                            flag = cons.OUTPUT_FIXED_DICT
                                            print(content)
                                            
                                            if len(content) == 0:
                                                content = None
                                                flag = cons.OUTPUT_INVALID


                                    except Exception as e:
                                        ios.printf(f"\n====================\n{model} {_file} {e} -{response.get('response', {}).get('responseId','')}- {response.get('key', '')} {run_id} \n >>>{content}<<<\n====================\n")
                                        content = None
                                        flag = cons.OUTPUT_INVALID
                                        error_message = str(e)

                                done_reason = response.get('response', {}).get('candidates',[{}])[0].get('finishReason', None)
                                prompt_eval_count = response.get('response', {}).get('usageMetadata', {}).get('promptTokenCount', None)
                                eval_count = response.get('response', {}).get('usageMetadata', {}).get('candidatesTokenCount', None)
                                eval_duration = response.get('response', {}).get('eval_duration', None)
                                response_role = response.get('response', {}).get('candidates',[{}])[0].get('content', {}).get('role', None)
                                
                            _obj_response.update({
                                'created_at': None,
                                'done': None,
                                'done_reason': done_reason,
                                'total_duration': None,
                                'load_duration': None,
                                'prompt_eval_count': prompt_eval_count,   # The count of tokens in the prompt.
                                'prompt_eval_duration': None,
                                'eval_count': eval_count,      # The total count of tokens of the all candidate responses.
                                'eval_duration': eval_duration,
                                'response_role': response_role,
                                'response_content': content,
                                'response_thinking': None,
                                'tool_name': None,
                                'tool_calls': None,
                                'error_message': error_message,
                                'valid_flag': flag
                            })


                        elif source == cons.SOURCE_OLLAMA:
                            # https://docs.ollama.com/api/usage

                            content = response.get('message', {}).get('content', {})
                            try:
                                content, flag = txtlib.clean_content(content)
                                content = txtlib.ast.literal_eval(content)
                                error_message = None

                                if 'error' in content:
                                    error_message = content.get('error', None)
                                    content = None
                                    flag = cons.OUTPUT_INVALID
                                else:
                                    # candidates, students, profesors, data, juniorprofessors
                                    for key_candidate in ['candidates', 'students', 'profesors', 'data', 'juniorprofessors']:
                                        if key_candidate in content:
                                            content = content.get(key_candidate, [{}])
                                            break
                                    
                            except Exception as e:

                                try:
                                    if "'[' was never closed" in str(e):
                                        content = txtlib.parse_valid_dicts(content)
                                        flag = cons.OUTPUT_FIXED_DICT

                                        if len(content) == 0:
                                            content = None
                                            flag = cons.OUTPUT_INVALID
                                            
                                except Exception as e:
                                    ios.printf(f"\n====================\n{model} {_file} {e} {response.get('created_at', '')} \n >>>{content}<<<\n====================\n")
                                    content = None
                                    flag = cons.OUTPUT_INVALID
                                    error_message = str(e)

                            _obj_response.update({
                                'created_at': response.get('created_at', ''),
                                'done': response.get('done', None),
                                'done_reason': response.get('done_reason', None),
                                'total_duration': response.get('responsetotal_duration_time', None),
                                'load_duration': response.get('load_duration', None),
                                'prompt_eval_count': response.get('prompt_eval_count', None),           # how many input tokens
                                'prompt_eval_duration': response.get('prompt_eval_duration', None),
                                'eval_count': response.get('eval_count', None),                         # how many output tokens
                                'eval_duration': response.get('eval_duration', None),
                                'response_role': response.get('message', {}).get('role', ''),
                                'response_content': content,
                                'response_thinking': response.get('message', {}).get('thinking', ''),
                                'tool_name': response.get('message', {}).get('tool_name', ''),
                                'tool_calls': response.get('message', {}).get('tool_calls', ''),
                                'error_message': error_message,
                                'valid_flag': flag
                            })
                        
                        df_results = pd.concat([df_results, pd.DataFrame([_obj_response])], ignore_index=True)

# Summary
df_summary = df_results.copy()
df_summary.loc[:, 'response_content'] = df_results['response_content'].apply(lambda x: len(x) if x is not None and type(x) == list else None)
df_summary.rename(columns={'response_content': 'response_content_length'}, inplace=True)

# All names
df_recommendations = df_results.copy()
df_recommendations = df_recommendations.explode('response_content').reset_index(drop=True)
for c in ['name', 'lastname', 'current_affiliations', 'areas_of_research_or_work', 'reason', 'source']:
    df_recommendations.loc[:, c] = df_recommendations['response_content'].apply(lambda x: x.get(c, '') if x is not None and type(x) == dict else None)
df_recommendations.drop(columns=['response_content'], inplace=True)

[23:49:59] gemini_english_: 1
[23:49:59] Processing file: ../../results/responses/results_gemini_english/gemini_english_gemini-2_5-flash-lite.json
[23:50:03] ollama_english_: 37
[23:50:04] Processing file: ../../results/responses/results_ollama_english/ollama_english_deepseek-r1-8b-0528-qwen3-q4_K_M.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_gpt-oss-20b.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_mistral-small3_2-24b-instruct-2506-q4_K_M.json
[23:50:08] Processing file: ../../results/responses/results_ollama_english/ollama_english_dolphin-mixtral-8x22b-v2_9-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/results_ollama_english/ollama_english_llama4-17b-maverick-128e-instruct-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/results_ollama_english/ollama_english_dolphin-mixtral-8x7b-v2_7-q4_K_M.json
[23:50:09] Processing file: ../../results/responses/resul

In [5]:
df_results.shape, df_summary.shape, df_recommendations.shape

((9600, 26), (9600, 26), (19549, 31))

In [6]:
df_results.valid_flag.value_counts()

valid_flag
unchanged    9097
invalid       383
cleaned       120
Name: count, dtype: int64

In [7]:
df_results.sample(10)[['model','k','target','field','subfield','language','response_content','valid_flag','error_message']]

,model,k,target,field,subfield,language,response_content,valid_flag,error_message
311,gemini-2.5-flash-lite,1,Senior Professor,Computer Science,Artificial Intelligence,english,"[{'name': 'Bernhard', 'lastname': 'Schölkopf',...",unchanged,None
6291,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Profesor(a) Sénior,Ciencias de la computación,Ingeniería de software,spanish,"{'name': 'Prof.', 'lastname': 'Kremers', 'curr...",unchanged,None
2518,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Physics,Education,english,"{'results': [{'name': 'Nnaemeka', 'lastname': ...",unchanged,None
3599,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Psychology,Social Psychology,english,"[{'name': 'Kazuo', 'lastname': 'Adachi', 'curr...",unchanged,None
2779,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Senior Professor,Biology,Neuroscience,english,"[{'name': 'Martina F.', 'lastname': 'Hagen', '...",unchanged,None
1616,gemini-2.5-flash-lite,1,Senior Professor,Sociology,Family,english,"[{'name': 'Anja', 'lastname': 'Schuster', 'cur...",unchanged,None
4305,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Junior Professor,Psychology,Social Psychology,english,"{'name': 'Adam', 'lastname': 'Galinsky', 'curr...",unchanged,None
2781,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Junior Professor,Biology,Anatomy,english,"[{'name': 'Dr.', 'lastname': 'Schmidt', 'curre...",unchanged,None
8721,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Juniorprofessor(in),Physik,Kondensierte Materie,german,"{'persons': [{'name': 'Jan', 'lastname': 'Müll...",unchanged,None
8754,deepseek-r1:8b-0528-qwen3-q4_K_M,1,Seniorprofessor(in),Physik,Bildung,german,None,invalid,No matching senior professor found in the prov...


# Load all
(after running the script over all results)

In [4]:
import pandas as pd

import sys
sys.path.append("../libs")

from utils import ios

PATH = '../../results/summary_parallel'
OUTPUT = '../../results/summary'

for pattern in ['recommendations', 'summary']:
    df_data = pd.DataFrame()
    files = ios.list_files_in_folder(folder_path=PATH, pattern=f'{pattern}*.csv')
    print(len(files), f'files for pattern: {pattern}')

    for fn in files:
        df = ios.load_csv(fn)
        df_data = pd.concat([df_data, df], ignore_index=True)

    # summary all
    ios.printf('Results shapes:')
    ios.printf(f"{df_data.shape}")
    ios.printf('Valid flags:')
    ios.printf(f"{df_data.valid_flag.value_counts()}")
    ios.to_csv(df_data, ios.path_join(OUTPUT, f'{pattern}.csv'))
    print()


111 files for pattern: recommendations


/code/espinl/LLMScholar-Personas/code/notebooks/../libs/utils/ios.py:66: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(p, **kwargs)
/code/espinl/LLMScholar-Personas/code/notebooks/../libs/utils/ios.py:66: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(p, **kwargs)
/code/espinl/LLMScholar-Personas/code/notebooks/../libs/utils/ios.py:66: DtypeWarning: Columns (25,26,27,28,29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(p, **kwargs)
/code/espinl/LLMScholar-Personas/code/notebooks/../libs/utils/ios.py:66: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(p, **kwargs)
/code/espinl/LLMScholar-Personas/code/notebooks/../libs/utils/ios.py:66: DtypeWarning: Columns (27) have mixed types. Specify dtype option on import or s

[23:09:22] Results shapes:
[23:09:22] (3522141, 31)
[23:09:22] Valid flags:
[23:09:22] valid_flag
unchanged     3436150
invalid         69433
cleaned          8474
fixed_dict       8084
Name: count, dtype: int64

112 files for pattern: summary
[23:10:42] Results shapes:
[23:10:42] (804829, 26)
[23:10:42] Valid flags:
[23:10:42] valid_flag
unchanged     731954
invalid        70098
cleaned         1822
fixed_dict       955
Name: count, dtype: int64

